# THERMA SEARCH AI · 07 · Kontrollü eğitim hazırlığı

**Çalışma zamanı → Tümünü çalıştır yalnız hazırlık ve GPU bellek preflight yapar. Eğitim varsayılan kapalıdır.** Google Drive izni gerekebilir. Checkpoint/veri indirilmez; eski run’lara yazılmaz; web değiştirilmez.

P07-C0 (640 kontrol), P07-A1 (scale=.25), P07-A2 (mosaic=0) ve P07-R1 (960 sistem çözünürlüğü) özgün HIT-UAV train/val ile hazırlanır. WiSARD ve HIT-UAV test açılmaz, model/threshold seçmez.

Faz 06 `run_phase06_20261127T152934Z_2071fdaa` mevcut raporunda 282 görüntü/1114 GT, mAP50=.8993, mAP50:95=.4925, fixed 996 TP/289 FP/118 FN ve both<16 recall 166/206 bildirilmiştir. **Bu notebook bu sayıları yeniden ölçmüş değildir.** Kaynak validation önceden model seçiminde kullanılmıştır; bağımsız final test değildir.

## 1 · Sabit kod ve paket
Ultralytics 8.3.228 korunur. Torch/torchvision zorla yeniden kurulmaz; gerçek ortam kaydedilir ve aynı kontrollü blok içinde değişmesine izin verilmez. Model eğitimi bu hücrede başlamaz.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY='https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF='feature/phase07-controlled-training'  # Teslimde sınanmış yardımcı commit hash’ine sabitlenir.
local_code_root=os.environ.get('THERMA_CODE_ROOT')
if local_code_root:
    PROJECT_ROOT=Path(local_code_root).resolve()
else:
    PROJECT_ROOT=Path('/content/therma-search-ai-phase07')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git','clone','--quiet',REPOSITORY,str(PROJECT_ROOT)],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'remote','get-url','origin'],text=True).strip()!=REPOSITORY:
        raise RuntimeError('Beklenmeyen kod deposu.')
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Yerel kod değişikliği var; üzerine yazılmadı. Temiz Colab çalışma zamanı kullanın.')
    subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','--depth','1','origin',CODE_REF],check=True)
    subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--upgrade-strategy','only-if-needed','-r',str(PROJECT_ROOT/'requirements-training-colab.txt')],check=True)
sys.path.insert(0,str(PROJECT_ROOT))
for module in list(sys.modules):
    if module=='src' or module.startswith('src.'):
        del sys.modules[module]
from src.training.phase07_plan import prepare_phase07, restore_data, preparation_report
from src.training.phase07_runtime import memory_preflight, run_block
from IPython.display import display, Image as DisplayImage
GIT_COMMIT=subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'],text=True).strip()
print('Kod revizyonu:',GIT_COMMIT)

## 2 · Drive ve çalışma politikası
Gerçek baseline `args.yaml`, etkin argümanlar, optimizer grupları, run özeti, ilk pretrained ağırlık ve veri manifesti **zorunludur**. Eksik ayar varsayılmaz; tanılama kaydıyla durulur. Eski best.pt başlangıç noktası olarak kullanılmaz: dört yeni kol aynı özgün pretrained yolo11s.pt SHA256 ile başlar.

`COMMON_BATCH` bütün kollar içindir. 960 bellek kontrolü geçmezse yeni bir blokta bütün kolların ortak batch’i düşürülmelidir. `RESUME_BLOCK` yalnız önceki Faz 07 hazırlık bloğuna dönmek içindir; checkpoint/raporların üzerine yazılmaz.

In [ ]:
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')
BASELINE_RUN=Path(os.environ.get('THERMA_PHASE07_BASELINE','/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_yolo11s/run_yolo11s_20261120T151540Z_78fb2af9'))
HIT_UAV_ZIP=Path(os.environ.get('THERMA_PHASE07_HIT_ZIP','/content/drive/MyDrive/ThermaSearchAI/datasets/hit_uav_v1.2.1.zip'))
OUTPUT_ROOT=Path(os.environ.get('THERMA_PHASE07_OUTPUT','/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_phase07'))
WORK_ROOT=Path(os.environ.get('THERMA_PHASE07_WORK','/content/therma-phase07-work'))
COMMON_BATCH=4
RESUME_BLOCK=os.environ.get('THERMA_PHASE07_BLOCK')  # Yoksa daima yeni, eşsiz blok.
ENABLE_TRAINING=False  # Bu fazda kapalı. Sonraki gerçek eğitim için açık kullanıcı yetkisi gerekir.
print('Hazırlık:',BASELINE_RUN,'Ortak batch:',COMMON_BATCH,'Eğitim:',ENABLE_TRAINING)

## 3 · Kanıt doğrulama ve yalnız train/val kopyası
DontCare içeren ve sıfır ham anotasyonlu görüntüler önceki politika ile dışlanır. Diğer sınıf kutuları kaldırılır; diğer-sınıf-only görüntüler önceki etiket tamlığı varsayımıyla Person arka planıdır. ZIP sadece okunur; test görüntü/JSON’u açılmaz. Yeniden üretilen train/val üyeliği, dışlama gerekçeleri, GT ve dosya hash’leri gerçek baseline manifestiyle bire bir karşılaştırılır.

In [ ]:
if RESUME_BLOCK:
    PHASE07_BLOCK=Path(RESUME_BLOCK).resolve()
    DATASET_ROOT=restore_data(PHASE07_BLOCK,HIT_UAV_ZIP,WORK_ROOT)
else:
    PHASE07_BLOCK,PREPARATION=prepare_phase07(BASELINE_RUN,HIT_UAV_ZIP,OUTPUT_ROOT,WORK_ROOT,
                                             git_commit=GIT_COMMIT,batch=COMMON_BATCH)
    DATASET_ROOT=Path(PREPARATION['dataset_directory'])
PLAN=json.loads((PHASE07_BLOCK/'frozen_plan.json').read_text())
from IPython.display import Markdown
print('Blok:',PHASE07_BLOCK)
display(Markdown(preparation_report(PHASE07_BLOCK).read_text()))

## 4 · Bütün kollar için GPU bellek preflight
Her aktif kol için iki gerçek augmented train batch’inde FP32 forward/loss/backward yapılır; **optimizer.step çağrılmaz, eğitim döngüsü çalışmaz**. AdamW moment buffer maliyeti ayrıca ayrılır. Trainer validation’ın 2×batch politikası için aynı çözünürlükte sentetik kare tensor üzerinde forward da ölçülür. En az %20 serbest bellek payı istenir; bu iki örnek tam epoch/OOM garantisi değildir.

GPU yoksa CPU eğitimi veya bellek testi başlamaz. Sonuç bloke olarak kaydedilir. OOM veya yetersiz payda yalnız bir kolun batch’i azaltılmaz; yeni bütün-kol bloğu gerekir. Preflight GPU/ortamı blok boyunca sabitler; farklı GPU ile yarım bloğa devam edilmez.

In [ ]:
PREFLIGHT_DIR=None
try:
    PREFLIGHT_DIR,MEMORY_PREFLIGHT=memory_preflight(PHASE07_BLOCK,DATASET_ROOT)
except RuntimeError as exc:
    MEMORY_PREFLIGHT={'status':'blocked','training_executed':False,'error':str(exc)}
print(json.dumps(MEMORY_PREFLIGHT,ensure_ascii=False,indent=2))
print('GPU belleği sadece başarılı GPU ölçümü varsa doğrulanmıştır.')

## 5 · Sonraki yetkili eğitim için kapalı yürütücü
Bu teslimde aşağıdaki kapı kapalı kalır. Gerçek eğitim ayrıca yetkilendirildikten sonra aynı notebook’ta `ENABLE_TRAINING=True` seçilebilir. Ortak optimizer/LR/scheduler/decay/batch/seed/veri/değerlendirme ayarları değiştirilemez. Her kol yeni attempt dizinine yazar; tamamlanan kol atlanır, yarım kalan kol hash ile mühürlenmiş son epoch snapshot’ından **yeni attempt** içinde sürdürülür. Eski snapshot ve checkpoint değiştirilmez.

Native resume EMA ve FP16 optimizer durumu kullanır; RNG/dataloader durumu tam geri yüklenmez, kesintisiz çalışmaya bit düzeyinde eşitlik iddia edilmez. Kesinti ve devam bilgisi sonuçlara dahildir. OOM tüm bloğu durdurur; ortak batch için yeni C0 dahil yeni blok gerekir. Stale `active.lock` varsa elle incelemeden otomatik devralınmaz.

In [ ]:
if ENABLE_TRAINING:
    if MEMORY_PREFLIGHT.get('status')!='passed' or PREFLIGHT_DIR is None:
        raise RuntimeError('Bütün kolların ortak GPU preflight kontrolü geçmeden eğitim başlayamaz.')
    ARM_RESULTS=run_block(PHASE07_BLOCK,DATASET_ROOT,PREFLIGHT_DIR,training_authorized=True)
    print(json.dumps(ARM_RESULTS,ensure_ascii=False,indent=2))
else:
    print('Hazırlık tamamlandı; gerçek GPU eğitimi başlatılmadı.')
print('Rapor:',PHASE07_BLOCK/'preparation_report.md')
print('Dondurulmuş plan:',PHASE07_BLOCK/'frozen_plan.json')
print('Bellek kontrolü:',PREFLIGHT_DIR)
print('WiSARD/test ile seçim yapılmaz. Final genelleme için yeni bağımsız dış veri gerekir.')